In [12]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(497128, 31)


In [13]:
interaction = (
    df.groupby(
        ["customerId", "skuNumber"]
    )["effective_qty"]
    .sum()
    .reset_index()
)

interaction.head()

,customerId,skuNumber,effective_qty
0,USR-100,SKU00010,1
1,USR-100,SKU00022,2
2,USR-100,SKU00026,1
3,USR-100,SKU00030,2
4,USR-100,SKU00060,6


In [14]:
print(
    "Interactions:",
    len(interaction)
)

print(
    "Retailers:",
    interaction["customerId"].nunique()
)

print(
    "Products:",
    interaction["skuNumber"].nunique()
)

Interactions: 115866
Retailers: 8640
Products: 252


In [15]:
matrix = interaction.pivot(
    index="customerId",
    columns="skuNumber",
    values="effective_qty"
).fillna(0)

matrix.shape

(8640, 252)

In [16]:
zeros = (matrix == 0).sum().sum()

total = (
    matrix.shape[0]
    * matrix.shape[1]
)

sparsity = (
    zeros / total
) * 100

print(
    f"Sparsity: {sparsity:.2f}%"
)

Sparsity: 94.68%


In [17]:
matrix.to_parquet(
    "../data/processed/interaction_matrix.parquet"
)

print("Saved")

Saved


In [18]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName",
            "brand",
            "category"
        ]
    ]
    .drop_duplicates()
)

sku_lookup.to_parquet(
    "../data/processed/sku_lookup.parquet",
    index=False
)

print("Saved")

Saved


In [19]:
top_products = (
    df.groupby("itemName")
    ["effective_qty"]
    .sum()
    .sort_values(
        ascending=False
    )
    .head(20)
)

top_products

itemName
Rajnigandha 17g 2 Pcs                    169975
Rajnigandha 4g                           133197
TZ 00 (with Silver) 0.45g                107882
Rajnigandha 17g Zipper 1 Pcs              32980
Rajnigandha 100g                          32152
TZ 00 4.25g (6 Pcs)                       30648
Tulsi 00 (with Silver) 0.45g              22749
RG Pearl Elaichi MP Pouch 0.14g           16973
RG Pearl Elaichi 1.4g Hanger              13138
Pass Pass Meetha Magic 11.75g Hanger      13090
Pass Pass Meetha Magic 2.2g (100 pcs)     11363
Rajnigandha 17g (1 Pcs)                   11159
Center Fresh                              10738
Center Fruit                               8575
Pulse Kachha Aam 175 Candy                 8286
Tulsi 00 4.25g (6 Pcs)                     8017
Jabsons Hing Jeera Peanuts                 7110
Bingo Tedhe Medhe                          7033
Haldiram - Salted Peanuts (24 pcs)         6688
Alpenliebe Creamfills Butter Toffee        6030
Name: effective_qty, dtype: int

In [20]:
matrix.shape

(8640, 252)

In [21]:
top_products.head(10)

itemName
Rajnigandha 17g 2 Pcs                   169975
Rajnigandha 4g                          133197
TZ 00 (with Silver) 0.45g               107882
Rajnigandha 17g Zipper 1 Pcs             32980
Rajnigandha 100g                         32152
TZ 00 4.25g (6 Pcs)                      30648
Tulsi 00 (with Silver) 0.45g             22749
RG Pearl Elaichi MP Pouch 0.14g          16973
RG Pearl Elaichi 1.4g Hanger             13138
Pass Pass Meetha Magic 11.75g Hanger     13090
Name: effective_qty, dtype: int64

In [22]:
matrix.shape

(8640, 252)